# Preparación de datos - G6

In [80]:
# Liberias
import pandas as pd
import numpy as np
from unidecode import unidecode
from datetime import date
import rasterio

## 1. Base de datos principal

- Casos confirmados de Dengue y Dengue grave mensual 2007 a 2024 agregado por municipio de residencia, obtenida del SISPRO del ministerio de salud.
- Codigo divipola y coordenadas obtenidas de datos del DANE
https://www.datos.gov.co/Mapas-Nacionales/DIVIPOLA-C-digos-municipios-geolocalizados/vafm-j2df/about_data
- Datos climaticos mesnuales de temperatura y precipitación obtenidos de archivos historicos de WorldClim
https://www.worldclim.org/data/monthlywth.html


In [81]:

#* Cargar datos de dengue
df = pd.read_csv('data/BaseDengue.csv', sep='|')

#* Remover numero y '-' en columnas Pais, Departamento y Municipio, y conversión a minusculas
cls = ['País', 'Departamento', 'Municipio']
for c in cls:
    df[c] = df[c].apply(lambda x: unidecode(x.split('-')[-1].strip().lower()))
df.rename(columns={'Año Epidemiologico': 'Year'}, inplace=True)
df.columns = [unidecode(c) for c in df.columns]

#* Eliminar pais no definido
df = df[df['Pais'] != 'no definido']

#* Eliminar municipios sin informacion
[i for i in df['Municipio'].unique() if 'sin informacion' in i]
df = df[~df['Municipio'].str.contains('sin informacion')]

#* Reordenar columnas
df = df.melt(
    id_vars=['Pais', 'Departamento', 'Municipio', 'Year'], # Columnas que se mantienen fijas
    var_name='Mes',              # Nombre para la nueva columna con los meses
    value_name='CasosConfirmados' # Nombre para la columna con los valores
)

#* leer archivo divipola
divipola = pd.read_excel('data/Listados_DIVIPOLA.xlsx', sheet_name='Municipios', skiprows=10, skipfooter=10)
divipola
divipola.columns = ['CodigoDepartamento', 'Departamento', 'CodigoMunicipio',
                    'Municipio',
                    'TipoMunicipio', 'Longitud', 'Latitud', 'Nota']

divipola.drop(columns=['Nota', 'CodigoDepartamento'], inplace=True)


#* decode Departamento, Municipio y TipoMunicipio
divipola['Departamento'] = divipola['Departamento'].apply(lambda x: unidecode(x.strip().lower()))
divipola['Municipio'] = divipola['Municipio'].apply(lambda x: unidecode(x.strip().lower()))
divipola['TipoMunicipio'] = divipola['TipoMunicipio'].apply(lambda x: unidecode(x.strip().lower()))

#* Corrección y estandarización de nombres de municipios en df para que coincidan con divipola
renamer = {
    'putumayo': {'leguizamo': 'puerto leguizamo'},
    'cauca': {'sotara': 'sotara paispamba',
              'piendamo': 'piendamo - tunia',
              'lopez': 'lopez de micay'},
    'magdalena': {'cerro san antonio': 'cerro de san antonio',
                  'chibolo': 'chivolo'},
    'antioquia': {'santafe de antioquia': 'santa fe de antioquia',
                  'san pedro': 'san pedro de uraba',
                  'san vicente': 'san vicente ferrer',
                  'don matias': 'donmatias'},
    'cordoba': {'san andres sotavento': 'san andres de sotavento',
                'purisima': 'purisima de la concepcion'},
    'guainia': {
                'mapiripana': 'mapiripan',
                'barranco minas': 'barrancominas'},
    'valle del cauca': {'cali': 'santiago de cali'},
    'cundinamarca': {'san juan de rio seco': 'san juan de rioseco'},
    'bolivar': {'cartagena': 'cartagena de indias',
                'mompos': 'santa cruz de mompox'},
    'sucre': {'tolu viejo': 'santiago de tolu'},
    'narino': {'cuaspud': 'cuaspud carlosama'},
    'tolima': {'mariquita': 'san sebastian de mariquita'},
    'amazonas': {'parana': 'miriti - parana'},
    'norte de santander': {'cucuta': 'san jose de cucuta'},
    'cesar': {'manaure': 'manaure balcon del cesar'},
    'vaupes': {'papunaua': 'papunahua'},
    'choco': {'belen de bajira': 'nuevo belen de bajira'}}

#* rename in df using renamer
for dept, munis in renamer.items():
    for muni, new_muni in munis.items():
        df.loc[(df['Departamento'] == dept) & (df['Municipio'] == muni), 'Municipio'] = new_muni

#* check Departamento and Municipio combinations present in df but not in divipola
df_combinations = set(zip(df['Departamento'], df['Municipio']))
divipola_combinations = set(zip(divipola['Departamento'], divipola['Municipio']))
missing_combinations = df_combinations - divipola_combinations
missing_combinations

#* delete in df where Departamento and Municipio are equal
df = df[~df.apply(lambda x: (x['Departamento'], x['Municipio']) in missing_combinations, axis=1)]

#* merge df and divipola on Departamento and Municipio
df = df.merge(divipola, on=['Departamento', 'Municipio'], how='left', )

#* reorder columns, drop pais and save to csv
df.drop(columns=['Pais'], inplace=True)
df.drop_duplicates(inplace=True)
df = df[['CodigoMunicipio','Departamento', 'Municipio','TipoMunicipio', 'Longitud', 'Latitud', 'Year', 'Mes', 'CasosConfirmados']]

#* Agregar datos climaticos y de elevacion
df['key'] = df['Departamento'] + '_' + df['Municipio']

#* unique combinations for Longitud and Latitud
locs = df[['key','Longitud', 'Latitud']].drop_duplicates(ignore_index=True)

In [82]:

#* obtener elevacion usando rasterio
files = {'Elevacion':r'data\worldclim\wc2.1_2.5m_elev\wc2.1_2.5m_elev.tif',
         } #! Datos elevacion
         
resultados = {}
for var,file in files.items():
    resultados[var] = []
    with rasterio.open(file) as src:
        # src.sample expects an iterable of (x, y) coordinates
        # For geospatial data, x is Longitude and y is Latitude
        for index, row in locs.iterrows():
            coordinates = [(row['Longitud'], row['Latitud'])]
            sample_gen = src.sample(coordinates)
            
            for result in sample_gen:
            # result is an array of values for each band. 
            # WorldClim files typically have 1 band per file.
                resultados[var].append(result[0])


In [83]:
locs['Elevacion'] = resultados['Elevacion']
#* agregar elevacion a df usando key
df = df.merge(locs[['key', 'Elevacion']], on='key', how='left')


In [84]:

#* Obetner temperatura y precipitacion mensual
files = {'tmin':[r'data\worldclim\wc2.1_cruts4.09_2.5m_tmin_2000-2009',
                           r'data\worldclim\wc2.1_cruts4.09_2.5m_tmin_2010-2019',
                           r'data\worldclim\wc2.1_cruts4.09_2.5m_tmin_2020-2024'],
         'tmax':[r'data\worldclim\wc2.1_cruts4.09_2.5m_tmax_2000-2009',
                           r'data\worldclim\wc2.1_cruts4.09_2.5m_tmax_2010-2019',
                           r'data\worldclim\wc2.1_cruts4.09_2.5m_tmax_2020-2024'],
         'prec':[r'data\worldclim\wc2.1_cruts4.09_2.5m_prec_2000-2009',
                         r'data\worldclim\wc2.1_cruts4.09_2.5m_prec_2010-2019',
                         r'data\worldclim\wc2.1_cruts4.09_2.5m_prec_2020-2024']
        }


locs = df[['key','Longitud', 'Latitud', 'Year', 'Mes']].drop_duplicates(ignore_index=True)
locs

,key,Longitud,Latitud,Year,Mes
0,antioquia_medellin,-75.581775,6.246631,2007,Enero
1,antioquia_medellin,-75.581775,6.246631,2008,Enero
2,antioquia_medellin,-75.581775,6.246631,2009,Enero
3,antioquia_medellin,-75.581775,6.246631,2010,Enero
4,antioquia_medellin,-75.581775,6.246631,2011,Enero
...,...,...,...,...,...
178351,vichada_cumaribo,-69.795533,4.446352,2020,Diciembre
178352,vichada_cumaribo,-69.795533,4.446352,2021,Diciembre
178353,vichada_cumaribo,-69.795533,4.446352,2022,Diciembre
178354,vichada_cumaribo,-69.795533,4.446352,2023,Diciembre


In [85]:
parse_month = {'Enero': '01', 'Febrero': '02', 'Marzo': '03', 'Abril': '04', 'Mayo': '05', 'Junio': '06',
               'Julio': '07', 'Agosto': '08', 'Septiembre': '09', 'Octubre': '10', 'Noviembre': '11', 'Diciembre': '12'}

locs['Mes'] = locs['Mes'].apply(lambda x: parse_month[x])
df['Mes'] = df['Mes'].apply(lambda x: parse_month[x])
locs.sort_values(['Year', 'Mes'], inplace=True)
locs.reset_index(drop=True, inplace=True)
#* por cada combinacion año mes, abrir archivo y obtener datos climaticos por coordenada
y_m_combinations = locs[['Year', 'Mes']].drop_duplicates()
results = []
for var, folders in files.items():
    for y,m in y_m_combinations.itertuples(index=False):
        #* determinar carpeta a abrir segun año
        if y <= 2009:
            folder = folders[0]
        elif y <= 2019:
            folder = folders[1]
        else:
            folder = folders[2]
        
        file = f"{folder + '\\' + '_'.join(folder.split('\\')[-1].split('_')[:-1]) + f'_{y}-{m}.tif'}"
        with rasterio.open(file) as src:
            for index, row in locs[(locs['Year'] == y) & (locs['Mes'] == m)].iterrows():
                coordinates = [(row['Longitud'], row['Latitud'])]
                sample_gen = src.sample(coordinates)
                
                for result in sample_gen:
                    results.append((row['Longitud'], row['Latitud'], y, m, var, result[0]))
        

In [86]:
results_df = pd.DataFrame(results, columns=['Longitud', 'Latitud', 'Year', 'Mes', 'Variable', 'Valor'])
#* melt results_df para tener columnas separadas para cada variable
results_df = results_df.pivot_table(index=['Longitud', 'Latitud', 'Year', 'Mes'], columns='Variable', values='Valor').reset_index()
#* merge con df para agregar variables climaticas
df = df.merge(results_df, on=['Longitud', 'Latitud', 'Year', 'Mes'], how='left')

In [ ]:

#* Rename cols prec, tmax y tmin a Precipitacion, TempMax y TempMin
df.rename(columns={'prec': 'Precipitacion', 'tmax': 'TempMax', 'tmin': 'TempMin'}, inplace=True)
#* drop key column
df.drop(columns=['key'], inplace=True)
#* save to csv
df.to_csv('data/dengue_data_v2.csv', index=False, sep='|')

In [92]:
df

,CodigoMunicipio,Departamento,Municipio,TipoMunicipio,Longitud,Latitud,Year,Mes,CasosConfirmados,Elevacion,Precipitacion,TempMax,TempMin
0,5001,antioquia,medellin,municipio,-75.581775,6.246631,2007,01,1.0,1609,66.900002,25.0,17.0
1,5001,antioquia,medellin,municipio,-75.581775,6.246631,2008,01,128.0,1609,88.599998,26.0,15.0
2,5001,antioquia,medellin,municipio,-75.581775,6.246631,2009,01,30.0,1609,137.500000,26.0,15.0
3,5001,antioquia,medellin,municipio,-75.581775,6.246631,2010,01,130.0,1609,18.400000,29.0,16.0
4,5001,antioquia,medellin,municipio,-75.581775,6.246631,2011,01,126.0,1609,110.900002,27.0,15.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
178900,99773,vichada,cumaribo,municipio,-69.795533,4.446352,2020,12,5.0,134,77.699997,33.0,23.0
178901,99773,vichada,cumaribo,municipio,-69.795533,4.446352,2021,12,1.0,134,75.099998,33.0,23.0
178902,99773,vichada,cumaribo,municipio,-69.795533,4.446352,2022,12,7.0,134,77.000000,32.0,22.0
178903,99773,vichada,cumaribo,municipio,-69.795533,4.446352,2023,12,8.0,134,79.599998,33.0,23.0


In [94]:
df['TipoMunicipio'].unique()

<ArrowStringArray>
['municipio', 'isla', 'area no municipalizada']
Length: 3, dtype: str